In [469]:
from build123d import *
from ocp_vscode import *
from enum import IntFlag
import math

import sys
# from types import SimpleNamespace

MODELS_PATH = "/Volumes/SSD 990 2TB/dev/build123d/"
if MODELS_PATH not in sys.path:
    sys.path.append(MODELS_PATH)

from models.model_pushbutton_jhf58 import ModelPushbuttonJHF58, ModelPushbuttonJHF58Cutout, ModelPushbuttonJHF58BigButton

models = True


configdict = {
    "screw": {
        "heat_insert_r": 3.4/2,
        "shaft_cutout_r": 2.6/2,
        "head_cutout_r": 4.5/2,
        "head_cutout_depth": 3,
        "filament_r": 1.75/2
    },
    "gimbal": {
        "inner": {
            "outer_r": 12.5,
            "height": 10,
            "inner_r": 8
        },
        "center": {
            "outer_r": 15.25,
            "height": 10,
            "inner_r": 12.75
        },
        "outer": {
            "outer_r": 18,
            "height": 10,
            "inner_r": 15.5
        },
        "ears_dia": 10,
        "ears_length": 7.5,
        "ring_gap": 0.25,
        "ring_thickness": 5
        
    },
    "shaft": {
        "lower": {
            "outer_r": 10,
            "inner_r": 6,
            "length": 15
        },
        # "upper_length": 20,
        # "lower_length": 25,
        "inner": {
            "outer_r": 5.8,
            "inner_r": 4,
            "length": 50
        },
    },
    "front_panel": {
        "screw_mount_r": 20,
        "x": 40,
        "y": 40,
        "z": 5,
        "slot_width": 25,
        "slot_height": 15
    },

    "radial_plate": {
        "x": 40,
        "y": 40,
        "z": 8,
        "rim_w": 12
    },

    "knob": {
        "outer_r": 15,
        "height": 22,
        "wall": 2
    },

    "neck": {
        "base_r": 14,
        "h": 25,
        "base_h": 5
    },

    "shoulder_r": lambda r: 2 * math.sqrt(r**2 - (config.gimbal.center.height/2)**2)
   
}

class AttrDict:
    def __init__(self, d):
        for key, value in d.items():
            if isinstance(value, dict):
                value = AttrDict(value)
            self.__dict__[key] = value

config = AttrDict(configdict)



In [470]:

mounting_hole_radius = config.shoulder_r(
    config.gimbal.outer.outer_r + config.gimbal.outer.height
    )/2+config.front_panel.screw_mount_r


def height(part):
    return part.bounding_box().size.Z

def top_of(part): 
    return part.bounding_box().max.Z

def bottom_of(part):
    return part.bounding_box().min.Z

def front_of(part):
    return part.bounding_box().min.Y

def back_of(part):
    return part.bounding_box().max.Y

def left_of(part):
    return part.bounding_box().min.X

def right_of(part):
    return part.bounding_box().max.X

def align_to_bottom(part, other):
    return bottom_of(part) - bottom_of(other)

def align_to_top(part, other):
    return top_of(part) - top_of(other)

def align_to_right(part, other):
    return right_of(part) - right_of(other)

def align_to_left(part, other):
    return left_of(part) - left_of(other)

def align_to_front(part, other):
    # other += Pos(0, front_of(other) - front_of(part), 0) * part
    return front_of(part) - front_of(other)

def align_to_back(part, other):
    # other += Pos(0, back_of(other) - back_of(part), 0) * part
    return back_of(part) - back_of(other)

def stack_up(tostack):
    stack = tostack[0]
    for part in tostack[1:]:
        top_z = stack.bounding_box().max.Z
        stack += Pos(0, 0, top_z + part.bounding_box().size.Z/2) * part
    return stack

def stack_left(tostack):
    stack = tostack[0]
    for part in tostack[1:]:
        left_x = stack.bounding_box().min.X
        offset = left_x - part.bounding_box().size.X/2
        stack += Pos(offset, 0, 0) * part
    return stack


  



In [471]:
class Anchor(IntFlag):
    CENTER = 0

    LEFT   = 1 << 0
    RIGHT  = 1 << 1

    BACK   = 1 << 2
    FRONT  = 1 << 3

    BOTTOM = 1 << 4
    TOP    = 1 << 5

TOP    = Anchor.TOP
BOTTOM = Anchor.BOTTOM
LEFT   = Anchor.LEFT
RIGHT  = Anchor.RIGHT
FRONT  = Anchor.FRONT
BACK   = Anchor.BACK
CENTER = Anchor.CENTER


In [472]:
class Positioner:

    @staticmethod
    def anchor_point(
        obj: Part,
        anchor: Anchor = CENTER,
    ) -> Vector:
        """
        Calculate the world-space point associated with a bbox anchor.
        """

        bb = obj.bounding_box()

        # X

        if anchor & LEFT:
            x = bb.min.X
        elif anchor & RIGHT:
            x = bb.max.X
        else:
            x = (bb.min.X + bb.max.X) / 2

        # Y

        if anchor & FRONT:
            y = bb.min.Y
        elif anchor & BACK:
            y = bb.max.Y
        else:
            y = (bb.min.Y + bb.max.Y) / 2

        # Z

        if anchor & BOTTOM:
            z = bb.min.Z
        elif anchor & TOP:
            z = bb.max.Z
        else:
            z = (bb.min.Z + bb.max.Z) / 2

        return Vector(x, y, z)

    @classmethod
    def attach(
        cls,
        obj: Part,
        obj_anchor: Anchor,
        target: Part,
        target_anchor: Anchor,
        offset=(0, 0, 0),
    ) -> Part:
        """
        Move obj so that:

            obj_anchor --> target_anchor
        """

        p_obj = cls.anchor_point(obj, obj_anchor)
        p_target = cls.anchor_point(target, target_anchor)

        translation = (
            p_target
            - p_obj
            + Vector(*offset)
        )

        return obj.moved(Location(translation))


# ======================================================================
# Fluent Wrapper
# ======================================================================

class Positioned:
    def __init__(self, part: Part):
        self.part = part

    def to(
        self,
        target: Part,
        my: Anchor,
        their: Anchor,
        offset=(0, 0, 0),
    ):
        self.part = Positioner.attach(
            self.part,
            my,
            target,
            their,
            offset,
        )

        return self


In [473]:
def build_gimbal_center_a():
    """Gimbal Center + shaft surround"""

    mine = config.gimbal.center
    screw = config.screw

    part = (Sphere(mine.outer_r) - Sphere(mine.inner_r)) & Box(2*mine.outer_r, 2*mine.outer_r, mine.height) # & Box(earcut, earcut, gc_h)
    # Heat insert cutouts
    part -= Plane.YZ * Pos(0, 0, mine.inner_r-1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MIN))
    part -= Plane.YZ * Pos(0, 0, -mine.inner_r+1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MAX))
    part -= Plane.XZ * Pos(0, 0, mine.inner_r-1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MIN))
    part -= Plane.XZ * Pos(0, 0, -mine.inner_r+1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MAX))
    part.color = Color("#87ceeb")
    return part

# show(build_gimbal_center_a())



In [474]:

def build_gimbal_inner_a():
    """Gimbal Inner Ring"""

    mine = config.gimbal.inner
    screw = config.screw
    
    part = (Sphere(mine.outer_r) - Cylinder(mine.inner_r, mine.height)) & Box(2*mine.outer_r, 2*mine.outer_r, mine.height) # & Box(earcut, earcut, gc_h)
    # Heat insert cutouts
    part -= Plane.YZ * Pos(0, 0, mine.inner_r-1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MIN))
    part -= Plane.YZ * Pos(0, 0, -mine.inner_r+1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MAX))
    part -= Plane.XZ * Pos(0, 0, mine.inner_r-1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MIN))
    part -= Plane.XZ * Pos(0, 0, -mine.inner_r+1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MAX))
    part.color = Color("#47ceeb")

    return part

# show(build_gimbal_inner_a())



In [475]:
def build_gimbal_outer_a():
    """Gimbal Outer Ring"""

    mine = config.gimbal.outer
    screw = config.screw
    panel = config.front_panel

    part = (Sphere(mine.outer_r) - Sphere(mine.inner_r)) & Box(2*mine.outer_r, 2*mine.outer_r, mine.height) # & Box(earcut, earcut, gc_h)
    # Heat insert cutouts
    part -= Plane.YZ * Pos(0, 0, mine.inner_r-1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MIN))
    part -= Plane.YZ * Pos(0, 0, -mine.inner_r+1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MAX))
    part -= Plane.XZ * Pos(0, 0, mine.inner_r-1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MIN))
    part -= Plane.XZ * Pos(0, 0, -mine.inner_r+1) * Cylinder(screw.filament_r, mine.outer_r - mine.inner_r + 2, align=(Align.CENTER, Align.CENTER, Align.MAX))
    part.color = Color("#47ceeb")

    # Front Panel Screw Mounts
    with PolarLocations(panel.screw_mount_r, 4, start_angle=45):
        part += Cylinder(screw.heat_insert_r+3, mine.height+2).translate((0, 0, 1))- Cylinder(screw.heat_insert_r, mine.height+2).translate((0, 0, 1))

    part.color = Color("#07ceeb")
    return part

# show(build_gimbal_outer_a())

In [476]:

def build_front_panel():
    """Front Panel"""

    mine = config.front_panel
    screw = config.screw
    panel = Box(mine.x, mine.y, mine.z)
    panel = fillet(panel.edges().filter_by(Axis.Z), radius=5)

    # mount_r = me["screw_mount_offset"]
    # go_r = config["gimbal"]["center_dia"]/2 + (config["gimbal"]["ring_gap"] * 2) + config["gimbal"]["ring_thickness"]

    # Mounting Holes
    with PolarLocations(mine.screw_mount_r, 4, start_angle=45):
        c1 = Cylinder(
            screw.head_cutout_r,
            screw.head_cutout_depth,
            align=(Align.CENTER, Align.CENTER, Align.MIN)
            ).translate((0, 0, mine.z/2 - screw.head_cutout_depth))
        bot_z = c1.faces().sort_by(Axis.Z)[0].center().Z
        depth = mine.z - screw.head_cutout_depth
        c2 = Cylinder(
            screw.shaft_cutout_r,
            depth,
            align=(Align.CENTER, Align.CENTER, Align.MAX)
            ).translate((0, 0, bot_z))
        panel -= c1 + c2

    # Slot Cutout
    panel -= extrude(SlotOverall(width=mine.slot_width, height=mine.slot_height), amount=mine.z).translate((0, 0, -mine.z/2))
    panel -= extrude(SlotOverall(width=mine.slot_width, height=mine.slot_height), amount=mine.z).translate((0, 0, -mine.z/2)).rotate(Axis.Z, 90)
    panel.color = Color("darkolivegreen")

    return panel

# show(build_front_panel())
    

In [477]:
# def build_xy_plate():
#     go_r = config["gimbal"]["center_r"] + (config["gimbal"]["ring_gap"] * 2) + config["gimbal"]["ring_thickness"]
#     fp = config["front_panel"]
#     model = ModelPushbuttonJHF58Cutout()
#     model_height = model.bounding_box().max.Z - model.bounding_box().min.Z
#     x, y, z = fp["x"], fp["y"], 6

#     panel = Box(x, y, z)
#     panel = fillet(panel.edges().filter_by(Axis.Z), radius=5)
#     # mount_r = config["front_panel"]["screw_mount_offset"]
#     button_r = config["gimbal"]["center_r"]/2
    
#     with PolarLocations(button_r, 4):
#         panel -= ModelPushbuttonJHF58Cutout().translate((0, 0, -z/2 + z/2))
#         panel += ModelPushbuttonJHF58().translate((0, 0, -z/2 + model_height/2-0.4))
 
#     return panel
    

In [478]:
def build_twist_plate():
    plate = Box(13.5, 25, 5)
    base = plate
    plate += Positioner.attach(Cylinder(10, 5), CENTER, plate, RIGHT)
    plate -= Positioner.attach(Cylinder(config.shaft.lower.inner_r, 5), CENTER, base, RIGHT)

    # - Cylinder(config.shaft.inner.outer_r, 5)
    # plate -= Positioner.attach(Cylinder(config.shaft.inner.outer_r, 5), CENTER, plate, RIGHT)
    backstop = Box(10, 10, 6) - ModelPushbuttonJHF58Cutout().translate((0, 0, -1.5))
    backstop_model = Box(10, 10, 6) + ModelPushbuttonJHF58().translate((0, 0, 1.2))

    if models:
        backstops = [
            Positioner.attach(backstop_model.rotate(Axis.Y, 90), TOP | LEFT | FRONT, plate, BOTTOM | LEFT | FRONT).translate((-2.3, 0, 0)),
            Positioner.attach(backstop_model.rotate(Axis.Y, 90), TOP | LEFT | BACK, plate, BOTTOM | LEFT | BACK).translate((-2.3, 0, 0))
        ]
    else:
        backstops = [
            Positioner.attach(backstop.rotate(Axis.Y, 90), TOP | LEFT | FRONT, plate, BOTTOM | LEFT | FRONT),
            Positioner.attach(backstop.rotate(Axis.Y, 90), TOP | LEFT | BACK, plate, BOTTOM | LEFT | BACK)
        ]
    plate += backstops
    return plate

# show(build_twist_plate())

In [479]:

def build_lower_shaft():
    mine = config.shaft.lower
    gimbal = config.gimbal.inner

    # ring = Cylinder(mine.outer_r+1, 2) - Box(mine.outer_r*2+4, mine.outer_r/2, 2) - Box(mine.outer_r/2, mine.outer_r*2+4, 2) - Cylinder(gimbal.inner_r-0.1, 2)

    shaft =  Cylinder(gimbal.inner_r-0.1, gimbal.height/3) - Cylinder(mine.inner_r, gimbal.height/3)
    main = Cylinder(mine.outer_r, mine.length) - Cylinder(mine.inner_r, mine.length)
    shaft = stack_up([main, shaft])
    # shaft += Pos(0, 0, -2) * ring
    # Twist Plate
    shaft += Positioner.attach(build_twist_plate(), TOP | RIGHT, shaft, BOTTOM | RIGHT)
    shaft.color = Color("darkslateblue")
    
    return shaft

# show(build_lower_shaft())



In [480]:
def build_radial_plate():
    mine = config.radial_plate
    panel = config.front_panel 
    screw = config.screw
    # go_r = config["gimbal"]["center_r"] + (config["gimbal"]["ring_gap"] * 2) + config["gimbal"]["ring_thickness"]
    # go_ir = go_r - config["gimbal"]["ring_thickness"]/2
    # go_h = config["gimbal"]["center_height"]
    # hi_r = config["screw"]["heat_insert_r"]

    # inner = mine.x - mine.rim_w
    inner_cutout = Box(mine.x - mine.rim_w, mine.y - mine.rim_w, mine.z)
    with PolarLocations(panel.screw_mount_r, 4, start_angle=45):
            inner_cutout -=  Cylinder(screw.heat_insert_r+3, mine.z)
    plate = Box(mine.x, mine.y, mine.z) - inner_cutout # Box(mine.x - mine.rim_w, mine.y - mine.rim_w, mine.z)
    plate = fillet(plate.edges().filter_by(Axis.Z), radius=5)

    if models:
        offset = 16.5
        switches = [
            ModelPushbuttonJHF58().rotate(Axis.X, 90).translate((0, offset, 0)),
            ModelPushbuttonJHF58().rotate(Axis.X, -90).translate((0, -offset, 0)),
            ModelPushbuttonJHF58().rotate(Axis.Y, 90).translate((-offset, 0, 0)),
            ModelPushbuttonJHF58().rotate(Axis.Y, -90).translate((offset, 0, 0)),
        ]
        plate += switches
    else:
        offset = 18.5
        cutouts = [
            ModelPushbuttonJHF58Cutout().rotate(Axis.X, 90).rotate(Axis.Y, 90).translate((0, offset, 0)),
            ModelPushbuttonJHF58Cutout().rotate(Axis.X, -90).rotate(Axis.Y, 90).translate((0, -offset, 0)),
            ModelPushbuttonJHF58Cutout().rotate(Axis.Y, 90).translate((-offset, 0, 0)),
            ModelPushbuttonJHF58Cutout().rotate(Axis.Y, -90).translate((offset, 0, 0)),
        ]
        plate -= cutouts
    # mount_r = config["front_panel"]["screw_mount_r"]
    
    with PolarLocations(panel.screw_mount_r, 4, start_angle=45):
        plate += Pos(0, 0, top_of(plate) + 2.5) * Cylinder(screw.heat_insert_r+3, 5)

    height = plate.bounding_box().size.Z

    with PolarLocations(panel.screw_mount_r, 4, start_angle=45):
        cutouts = stack_up([
            Cylinder(screw.head_cutout_r, height-2.5),
            Cylinder(screw.shaft_cutout_r, 2.5)
        ])
        plate -= Pos(0, 0, align_to_bottom(plate, cutouts)) * cutouts
    plate.color = Color("darkgreen")
    return plate

# show(build_radial_plate())

In [481]:
def build_inner_shaft():
    mine = config.shaft.inner
    screw = config.screw

    wings = Box(20, 5, 7.5)
    cutout = stack_up([Cylinder(mine.inner_r, mine.length), Cylinder(screw.head_cutout_r, screw.head_cutout_depth), Cylinder(screw.shaft_cutout_r, 3)])
    wire_cutout = Cylinder(2, 20).rotate(Axis.Y, -30)
    part = Cylinder(mine.outer_r, height(cutout))
    dcutout = Box(2, mine.inner_r*2+2, 25)
    part -= Pos(align_to_right(part, dcutout), 0, align_to_top(part, dcutout)) * dcutout
    part -= Pos(-2, 0, align_to_top(part, wire_cutout)) * wire_cutout

    # part = Cylinder(config["shaft"]["inner_dia"]/2-1, config["shaft"]["inner_length"])
    part += Pos(0, 0, align_to_bottom(part, wings)) * wings
    part -= Pos(0, 0, align_to_bottom(part, cutout)) * cutout
    # part += Pos(0, 0, align_bottom_of(part, Cylinder(mine.inner_r, mine.length))) * Cylinder(mine.inner_r, mine.length)
    part.color = Color("darkred")
    return part

# show(build_inner_shaft())

In [482]:
def build_knob_neck():
    shaft = config.shaft.inner
    screw = config.screw
    mine = config.neck
    # knob_r = 20
    # knob_h = 30
    neck_inner_r = shaft.outer_r + 0.1
    neck_outer_r = neck_inner_r + 0.8
    base_r = mine.base_r
    neck_h = mine.h
    base_h = mine.base_h

    dkey = Box(2, shaft.inner_r*2, neck_h, align=(Align.CENTER, Align.CENTER, Align.MIN))

    # Neck
    build0 = Rectangle((neck_outer_r - neck_inner_r)*2, neck_h, align=(Align.MIN, Align.MIN)).translate((neck_inner_r, 0))
    build0 = revolve(build0, Axis.Y, 315).rotate(Axis.X, 90).rotate(Axis.Z, 180+45/2) 
    build0 += Pos(align_to_right(build0, dkey) - 1.4, 0, align_to_top(build0, dkey)) * dkey

    # Base
    build1 = Cylinder(base_r, base_h) - Cylinder(screw.heat_insert_r , base_h)
    build1 -= Pos(4+align_to_left(build1, Cylinder(3, base_h)), 0, 0) * Cylinder(2, base_h+4).rotate(Axis.Y, -30)
    with PolarLocations(10, 2, start_angle=90):
        build1 -= Cylinder(screw.shaft_cutout_r, build1.bounding_box().size.Z)

    # Switch Stack
    switch_cutout = ModelPushbuttonJHF58Cutout()
    print(switch_cutout.clearance)
    # cutout_stack = stack_up(
    #     [Box(switch_cutout.body_x + 2, switch_cutout.body_y + 2, switch_cutout.body_z + 2),
    #      Pos(0, 0, -switch_cutout.leg_z + 12) * switch_cutout
    #      ])
    # cutout_stack += Positioner.attach(Box(5, 5, 5), BOTTOM | RIGHT, cutout_stack, BOTTOM | LEFT)
    # return cutout_stack
    cyl_r = max(switch_cutout.body_x, switch_cutout.body_y)/2+3
    switch_stack = stack_up([
        Cylinder(cyl_r, 5) - Cylinder(cyl_r-1.5, 15),
        Cylinder(cyl_r, switch_cutout.body_z + switch_cutout.leg_z-2)
    ])
    switch_stack -= Pos(0, 0, align_to_top(switch_stack, switch_cutout)) * switch_cutout
    switch_stack -= Pos(-5, 0, align_to_bottom(switch_stack, Box(5, 5, 5))) * Box(5, 5, 5)

    part = stack_up([build0, build1, switch_stack.translate((0, 0, -5))])
    # part = stack_up([switch_stack])

    if models:
        part += Pos(0, 0, 38.5) * stack_up([ModelPushbuttonJHF58(), Pos(0, 0, -3) * ModelPushbuttonJHF58BigButton()])

    return part

# show(build_knob_neck())


In [483]:
def build_knob():
    def chord_radius(chord, height):
        return chord**2 / (8 * height) + height / 2
    
    bigbutton = ModelPushbuttonJHF58BigButton()
    mine = config.knob
    neck = config.neck

    chord_height = 2
    dome_length = mine.outer_r - bigbutton.bounding_box().size.X/2
    radius = chord_radius(dome_length, chord_height)
    chord = Circle(radius) - Pos(0, -2) * Rectangle(dome_length * 2, radius * 2)

    template = Rectangle(dome_length, mine.height)
    template = Pos(0, align_to_back(template, chord)) * chord
    template += Pos(0, align_to_back(template, rect := Rectangle(dome_length, mine.height))-chord_height) * rect
    rect = Rectangle(dome_length - mine.wall, mine.height-mine.wall)
    # Cutout the main part
    template -= Pos(align_to_left(template, rect), align_to_front(template, rect)) * rect
    # Lower Screw plate to neck
    rect = Rectangle(7, neck.base_h)
    template += Pos(align_to_right(template, rect) - mine.wall, align_to_front(template, rect) + neck.base_h) * rect
    template = Pos(bigbutton.bounding_box().size.X/2 + 0.5, 0) * template

    part = revolve(template, Axis.Y, 360)
    # part = template

    return part

# show(build_knob())

In [484]:
def show_integrated():
    # show(build_gimbal_center(), build_gimbal_inner_ring())
    # show(build_gimbal_center(), build_gimbal_inner_ring(), build_gimbal_outer_ring())
    show(build_gimbal_center_a(),
         build_gimbal_inner_a(),
         build_gimbal_outer_a(),
         build_front_panel().translate((0, 0, 9.5)),
         build_lower_shaft().translate((0, 0, -12.8)),
         build_radial_plate().translate((0, 0, -14.1)),
         build_inner_shaft().rotate(Axis.Z, 90).translate((0, 0, -5)),
         build_knob_neck().rotate(Axis.Z, 90).translate((0, 0, -1.8)),
         build_knob().rotate(Axis.X, 90).translate((0, 0, 38)),
        #  build_xy_plate().translate((0, 0, -20)))
    )

def packed():
    return pack([
                 build_front_panel(),
                 build_lower_shaft(),
                 build_radial_plate(),
                 build_inner_shaft(),
                 build_knob_neck(),
                 ], padding=10, align_z=True)

gimbals = Compound(children=[build_gimbal_center_a(), build_gimbal_inner_a(), build_gimbal_outer_a()])

# show(gimbals)

show_integrated()
# show(
#     build_gimbal_center_a(),
#          build_gimbal_inner_a(),
#          build_gimbal_outer_a())
# show(build_gimbal_outer_a())
# show(build_radial_plate())
# export_step(Compound(children=packed()), "/Volumes/SSD 990 2TB/Documents/stl/joystick/packed.step")
# print(packed())
export_step(Compound(children=[gimbals]), "/Volumes/SSD 990 2TB/Documents/stl/joystick/gimbals.step")
export_step(Compound(packed()), "/Volumes/SSD 990 2TB/Documents/stl/joystick/packed.step")




0.254


Standard_Failure: BRep_API: command not done